In [15]:
# ============================================
# CELDA 1: Reset del documento
# ============================================
doc_reset(hard=True)
print("Documento reseteado")

Documento reseteado


In [16]:
# ============================================
# CELDA 2: Portada y metadatos
# ============================================
from docx.shared import Inches, Cm, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime

with build_doc(block_id="portada", order=10) as builder:
    doc = builder.document
    
    # Metadatos del documento
    builder.metadata(
        title="Memoria de Cálculo - Viga Simplemente Apoyada",
        subject="Análisis Estructural",
        keywords=["ingeniería", "estructura", "viga", "flexión"]
    )
    
    # Configurar márgenes
    section = doc.sections[0]
    section.top_margin = Cm(2.54)
    section.bottom_margin = Cm(2.54)
    section.left_margin = Cm(2.54)
    section.right_margin = Cm(2.54)
    
    # Espaciado para centrar portada verticalmente
    for _ in range(6):
        p = doc.add_paragraph()
        p.paragraph_format.space_after = Pt(0)
    
    # Título principal
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run("MEMORIA DE CÁLCULO")
    run.bold = True
    run.font.size = Pt(28)
    run.font.color.rgb = RGBColor(0, 51, 102)
    run.font.name = "Calibri"
    
    # Subtítulo
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_before = Pt(6)
    run = p.add_run("Viga Simplemente Apoyada con Carga Distribuida")
    run.font.size = Pt(16)
    run.font.color.rgb = RGBColor(80, 80, 80)
    run.font.name = "Calibri"
    
    # Línea decorativa
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_before = Pt(12)
    run = p.add_run("━" * 40)
    run.font.color.rgb = RGBColor(0, 102, 153)
    run.font.size = Pt(14)
    
    # Información del proyecto
    for _ in range(2):
        doc.add_paragraph()
    
    info_lines = [
        ("Proyecto:", "Edificio Residencial El Mirador"),
        ("Ubicación:", "Santiago, Chile"),
        ("Elemento:", "Viga VP-01, Eje B, Nivel 3"),
        ("Fecha:", datetime.now().strftime("%d de %B de %Y")),
        ("Revisión:", "Rev. 0"),
    ]
    
    for label, value in info_lines:
        p = doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run_label = p.add_run(label + "  ")
        run_label.bold = True
        run_label.font.size = Pt(11)
        run_label.font.name = "Calibri"
        run_val = p.add_run(value)
        run_val.font.size = Pt(11)
        run_val.font.name = "Calibri"
        p.paragraph_format.space_after = Pt(2)

print("✅ Portada generada")

✅ Portada generada


In [17]:
# ============================================
# CELDA 3: Índice + Introducción + Normativa
# ============================================
with build_doc(block_id="introduccion", order=20) as builder:
    doc = builder.document
    
    builder.page_break()
    
    # Tabla de contenidos
    builder.heading("Índice", level=1)
    builder.table_of_contents(depth=2)
    builder.page_break()
    
    # Introducción
    builder.heading("1. Introducción", level=1)
    builder.text(
        "La presente memoria de cálculo tiene por objeto verificar la capacidad "
        "resistente y las deformaciones de la viga VP-01, correspondiente al eje B "
        "del nivel 3 del proyecto Edificio Residencial El Mirador. La viga se modela "
        "como simplemente apoyada y sometida a carga distribuida uniforme.",
        align="justify"
    )
    
    # Normativa
    builder.heading("2. Normativa Aplicable", level=1)
    builder.list([
        "NCh 427 - Acero estructural para uso general",
        "NCh 2369 - Diseño sísmico de estructuras industriales",
        "AISC 360-22 - Specification for Structural Steel Buildings",
        "ACI 318-19 - Building Code Requirements for Structural Concrete",
    ])

print("✅ Introducción y normativa generadas")

✅ Introducción y normativa generadas


In [18]:
# ============================================
# CELDA 4: Datos de entrada y propiedades
# ============================================
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT

# --- Datos del problema ---
L = 6.0        # Longitud [m]
w = 15.0       # Carga distribuida [kN/m]
b_sec = 0.20   # Ancho sección [m]
h_sec = 0.40   # Altura sección [m]
E = 200e6      # Módulo de elasticidad [kN/m²]
fy = 250e3     # Fluencia [kN/m²]

# Propiedades de sección
I = (b_sec * h_sec**3) / 12
S = (b_sec * h_sec**2) / 6
A = b_sec * h_sec

with build_doc(block_id="datos_entrada", order=30) as builder:
    doc = builder.document
    
    builder.heading("3. Datos de Entrada", level=1)
    builder.heading("3.1. Geometría y Materiales", level=2)
    
    builder.text(
        "A continuación se presentan los datos de entrada considerados "
        "para el análisis de la viga VP-01:",
        align="justify"
    )
    
    # Tabla de datos de entrada con formato nativo
    table = doc.add_table(rows=7, cols=4)
    table.style = "Table Grid"
    table.autofit = True
    
    # Encabezado de tabla
    headers = ["Parámetro", "Símbolo", "Valor", "Unidad"]
    for i, h in enumerate(headers):
        cell = table.cell(0, i)
        cell.text = h
        p = cell.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = p.runs[0]
        run.bold = True
        run.font.size = Pt(10)
        run.font.name = "Calibri"
        run.font.color.rgb = RGBColor(255, 255, 255)
        # Fondo azul oscuro
        from docx.oxml import OxmlElement
        from docx.oxml.ns import qn
        shading = OxmlElement('w:shd')
        shading.set(qn('w:fill'), '003366')
        shading.set(qn('w:val'), 'clear')
        cell._tc.get_or_add_tcPr().append(shading)
    
    # Datos
    data_rows = [
        ("Longitud de viga", "L", f"{L:.1f}", "m"),
        ("Carga distribuida", "w", f"{w:.1f}", "kN/m"),
        ("Ancho de sección", "b", f"{b_sec*100:.0f}", "cm"),
        ("Altura de sección", "h", f"{h_sec*100:.0f}", "cm"),
        ("Módulo de elasticidad", "E", f"{E/1e6:.0f}", "GPa"),
        ("Esfuerzo de fluencia", "fy", f"{fy/1e3:.0f}", "MPa"),
    ]
    
    for row_idx, (param, sym, val, unit) in enumerate(data_rows, 1):
        table.cell(row_idx, 0).text = param
        # Símbolo en itálica
        p = table.cell(row_idx, 1).paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = p.add_run(sym)
        run.italic = True
        run.font.size = Pt(10)
        # Valor centrado
        p = table.cell(row_idx, 2).paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.add_run(val).font.size = Pt(10)
        # Unidad centrada
        p = table.cell(row_idx, 3).paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.add_run(unit).font.size = Pt(10)
        # Formato de la primera columna
        table.cell(row_idx, 0).paragraphs[0].runs[0].font.size = Pt(10)
        # Fila alternante
        if row_idx % 2 == 0:
            for col_idx in range(4):
                shading = OxmlElement('w:shd')
                shading.set(qn('w:fill'), 'E8F0FE')
                shading.set(qn('w:val'), 'clear')
                table.cell(row_idx, col_idx)._tc.get_or_add_tcPr().append(shading)
    
    # Propiedades de sección
    builder.heading("3.2. Propiedades de la Sección", level=2)
    
    # Ecuación del momento de inercia
    p = doc.add_paragraph()
    p.add_run("Momento de inercia:  ").bold = True
    math_xml = builder.create_math_element(f"I = frac(b * h^3, 12) = frac({b_sec} * {h_sec}^3, 12) = {I*1e6:.2f} * 10^(-6)")
    p._p.append(math_xml)
    p.add_run(" m⁴")
    
    # Módulo de sección
    p = doc.add_paragraph()
    p.add_run("Módulo de sección:  ").bold = True
    math_xml = builder.create_math_element(f"S = frac(b * h^2, 6) = {S*1e6:.2f} * 10^(-6)")
    p._p.append(math_xml)
    p.add_run(" m³")
    
    # Área
    p = doc.add_paragraph()
    p.add_run("Área transversal:  ").bold = True
    math_xml = builder.create_math_element(f"A = b * h = {A*1e4:.0f}")
    p._p.append(math_xml)
    p.add_run(" cm²")

print(f"✅ Datos de entrada: L={L}m, w={w}kN/m, sección {b_sec*100:.0f}x{h_sec*100:.0f}cm")
print(f"   I = {I*1e6:.4f} x10⁻⁶ m⁴, S = {S*1e6:.4f} x10⁻⁶ m³, A = {A*1e4:.0f} cm²")

✅ Datos de entrada: L=6.0m, w=15.0kN/m, sección 20x40cm
   I = 1066.6667 x10⁻⁶ m⁴, S = 5333.3333 x10⁻⁶ m³, A = 800 cm²


In [19]:
# ============================================
# CELDA 5: Análisis estructural
# ============================================
with build_doc(block_id="analisis", order=40) as builder:
    doc = builder.document
    
    builder.heading("4. Análisis Estructural", level=1)
    builder.heading("4.1. Esfuerzos Máximos", level=2)
    
    builder.text(
        "Al estar sometida a flexión simple mediante carga distribuida, "
        "los esfuerzos máximos se determinan según las fórmulas clásicas de "
        "la resistencia de materiales, obteniendo reacción máxima en los apoyos "
        "y momento flector máximo en el centro del claro.",
        align="justify"
    )
    
    # Cálculos
    R = (w * L) / 2
    V_max = R
    M_max = (w * L**2) / 8
    
    builder.math(f"V_[max] = frac(w * L, 2) = frac({w:.1f} * {L:.1f}, 2) = {V_max:.2f} text( kN)", label="eq:shear")
    builder.math(f"M_[max] = frac(w * L^2, 8) = frac({w:.1f} * {L:.1f}^2, 8) = {M_max:.2f} text( kN*m)", label="eq:moment")
    
    doc.add_paragraph()
    p = doc.add_paragraph("Según la ecuación ")
    p.add_run("(1)").bold = True
    p.add_run(" el corte máximo es ")
    p.add_run(f"V = {V_max:.2f} kN").bold = True
    p.add_run(". Según la ecuación ")
    p.add_run("(2)").bold = True
    p.add_run(" el momento máximo de diseño es ")
    p.add_run(f"M = {M_max:.2f} kN·m").bold = True
    p.add_run(".")
    
print(f"✅ Análisis: Vmax = {V_max:.2f} kN, Mmax = {M_max:.2f} kN·m")

✅ Análisis: Vmax = 45.00 kN, Mmax = 67.50 kN·m


In [20]:
# ============================================
# CELDA 5: Análisis de verificación - Diseño
# ============================================

# Cálculos Lógicos (Fuera del constructor DOCX para poder usarlos en el texto)
sigma_max = M_max / S            # Esfuerzo flexión en kN/m²
tau_max = (3 * V_max) / (2 * A)  # Corte

# Convertir a MPa
sigma_mpa = sigma_max / 1e3
tau_mpa = tau_max / 1e3
fy_mpa = fy / 1e3

with build_doc(block_id="verificaciones", order=50) as builder:
    doc = builder.document
    
    # 5.1. Flexión
    builder.heading("5. Verificaciones de Diseño", level=1)
    builder.heading("5.1. Esfuerzo por Flexión", level=2)
    
    # Ecuación a mostrar
    doc.add_paragraph("Basado en el momento flector máximo en centro de claro, la tensión de fibra máxima esperada es:")
    builder.math(f"sigma_[max] = frac(M_[max], S) = frac({M_max:.2f}, {S*1e6:.2f} * 10^(-6)) = {sigma_mpa:.2f} text( MPa)")

    if sigma_mpa <= fy_mpa:
        # Párrafo exitoso con texto OK
        p = doc.add_paragraph()
        run = p.add_run(f"La tensión aplicada ({sigma_mpa:.1f} MPa) es menor al límite elástico del material de {fy_mpa:.1f} MPa.")
        p.add_run(" (CUMPLE)").font.bold = True
        p.runs[-1].font.color.rgb = RGBColor(0, 128, 0)
    else:
        # Párrafo con alerta de falla ROJO
        p = doc.add_paragraph()
        run = p.add_run(f"La tensión máxima aplicada de {sigma_mpa:.1f} MPa EXCEDE la tensión de fluencia de {fy_mpa:.1f} MPa. Se requiere aumentar la sección.")
        run.bold = True
        run.font.color.rgb = RGBColor(255, 0, 0)

    # 5.2. Corte
    # Fórmulas de esfuerzo transversal
    builder.heading("5.2. Esfuerzo de Corte por Esfuerzo Transversal", level=2)
    builder.text("Las tensiones principales obtenidas por efecto corte máximo son las siguientes:", align="justify")
    
    builder.math(f"tau_[max] = frac(3 * V_[max], 2 * A) = {tau_mpa:.3f} text( MPa)", label="eq:shear_dist")
    
    print(f"✅ Flexión ({sigma_mpa:.2f} MPa vs {fy_mpa} MPa) | Cortante ({tau_mpa:.2f} MPa)")

✅ Flexión (12.66 MPa vs 250.0 MPa) | Cortante (0.84 MPa)


In [21]:
# ============================================
# CELDA 6: Deflexión y Conclusión
# ============================================

# Cálculos 
delta_max = (5 * w * L**4) / (384 * E * I)  # en metros
delta_mm = delta_max * 1000

# Limite L/360
limit = L / 360
limit_mm = limit * 1000

with build_doc(block_id="deflexion", order=60) as builder:
    doc = builder.document
    
    # 6. Deflexión elástica
    builder.heading("6. Verificación de Deflexión", level=1)
    
    doc.add_paragraph("La carga produce deflexión elástica en régimen lineal debido a que todo el esfuerzo se ha validado de estar en el rango elástico por:")
    builder.math(f"delta = frac(5 * w * L^4, 384 * E * I) = {delta_mm:.3f} text( mm)")
    
    builder.heading("7. Criterio de Falla por Deflexión", level=1)
    builder.text(f"La deflexión máxima es de {delta_mm:.3f} mm, versus un límite general estructuralizado exigido de L/360 de {limit_mm:.2f} mm.")
    
    # 8. Conclusiones finales
    builder.heading("8. Conclusión General", level=1)
    
    if (delta_max <= limit) and (sigma_mpa <= fy_mpa):
         conclu = doc.add_paragraph()
         conclu.add_run("La viga VP-01 ").bold = True
         conclu.add_run("HA PASADO SATISFACTORIAMENTE ESTRECHAMENTE").font.color.rgb = RGBColor(0, 150, 0)
         conclu.add_run(" todas las validaciones.").bold = True
    else:
         conclu = doc.add_paragraph()
         conclu.add_run("La viga ").bold = True
         conclu.add_run("NO HA CUMPLIDO").font.color.rgb = RGBColor(255, 0, 0)
         conclu.add_run(" los requerimientos de diseño.").bold = True

    # Fin del docx
    builder.page_break()
    p_nota = doc.add_paragraph("Este cálculo fue generado de manera automática utilizando INSPYRO y algoritmos numéricos validados.")
    p_nota.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p_nota.runs[0].font.size = Pt(8)

print(f"✅ Documento completo finalizado. Deflexión: {delta_mm:.3f} mm (límite {limit_mm:.1f})")

✅ Documento completo finalizado. Deflexión: 1.187 mm (límite 16.7)
